# Semana 13 — Redes neurais em manutenção preditiva: escolha, preparo e avaliação

Este notebook é um laboratório passo a passo. Vamos começar com uma revisão curta da MLP da Semana 12, analisar um dataset público de manutenção, construir um classificador tabular com baseline e, depois, preparar uma série temporal pública para discutir CNN 1D, LSTM e autoencoders.

**Pergunta FactoryHub:** como transformar sinais de sensores em uma informação útil para priorizar inspeções, sem confundir um score estatístico com diagnóstico de falha?

## Objetivos de aprendizagem

- reconhecer quando a pergunta pede classificação, regressão ou detecção de anomalias;
- preparar variáveis sem incluir identificadores ou informações do futuro;
- treinar e avaliar uma MLP comparada com baseline;
- construir janelas temporais sem embaralhar a ordem;
- justificar MLP, CNN 1D, LSTM ou autoencoder pelo tipo de dado e pelo uso operacional.

## Como a rede neural se encaixa nas fases do CRISP-DM

O **CRISP-DM tem seis fases**. A rede neural ? uma t?cnica de modelagem dentro do processo, n?o uma fase do CRISP-DM.

| Fase do CRISP-DM | Aplica??o neste estudo |
|---|---|
| 1. Entendimento do neg?cio | definir decis?o de manuten??o, alvo, usu?rio e custo dos erros |
| 2. Entendimento dos dados | conhecer datasets, colunas, distribui??es e limita??es |
| 3. Prepara??o dos dados | selecionar entradas, evitar vazamento, criar atributos e parti??es |
| 4. Modelagem | construir pipeline e **treinar a MLP**; comparar com Random Forest ou CNN temporal |
| 5. Avalia??o | escolher limiar na valida??o, medir no teste e decidir se atende ao objetivo |
| 6. Implanta??o | documentar, disponibilizar sa?da ao FactoryHub e acompanhar o modelo |

Dentro das fases 3, 4 e 5 h? **passos de aprendizado de m?quina**: preparar X/y, separar dados, ajustar transforma??es, treinar, comparar, selecionar limiar e medir. Esses passos n?o s?o fases adicionais do CRISP-DM. O ciclo ? iterativo: uma avalia??o ruim pode levar de volta ? compreens?o dos dados ou ? prepara??o.

## Revis?o da Semana 12 ? MLP e CRISP-DM

Responda antes de seguir:

1. Qual ? a diferen?a entre classificar falha/sem falha e estimar temperatura futura?
2. Para que separar treino, valida??o e teste?
3. O que ? vazamento quando uma vari?vel s? existe depois da falha?
4. Por que comparar a MLP com um modelo de refer?ncia?

**S?ntese:** classifica??o estima classe; regress?o estima valor cont?nuo. Treino ajusta par?metros, valida??o escolhe configura??o/limiar e teste estima desempenho final. Usar informa??o posterior ? vazamento. O baseline mostra se a complexidade extra agrega valor.


## Dois datasets públicos, duas situações de aprendizagem

| Dataset | Origem e tipo | Bom para | Limite didático |
|---|---|---|---|
| AI4I 2020 Predictive Maintenance | UCI; 10 mil registros sintéticos de processo | classificação tabular e pipeline MLP | não representa todos os desafios de uma planta real; as linhas não formam uma série temporal confiável |
| MetroPT-3 | UCI; telemetria real de compressor ferroviário, multivariada e temporal | janelas, previsão de sinal e discussão de anomalias sequenciais | arquivo grande (~208 MB) e sem rótulo de falha em cada linha; exige cuidado com eventos e divisão temporal |

Vamos executar o estudo supervisionado no AI4I. O MetroPT-3 será usado em um segundo roteiro para preparar sequências e escolher arquitetura. Não compararemos diretamente as métricas dos dois datasets: tarefa e rótulos são diferentes.

## Passo 0 — Ambiente e reprodutibilidade

O código usa Python, pandas, NumPy e scikit-learn. `ucimlrepo` baixa os dados diretamente do UCI; a primeira execução precisa de internet. Se estiver em ambiente sem acesso à internet, baixe o CSV na página oficial e carregue-o localmente.

In [ ]:
# Se necessário, execute uma vez no terminal ou descomente no notebook:
# %pip install ucimlrepo pandas numpy scikit-learn matplotlib

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

SEED = 42
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

## Fase 1 do CRISP-DM — Entendimento do negócio (CRISP-DM)

**Decisão proposta:** priorizar uma inspeção quando as condições observadas se parecem com registros associados a falha no AI4I.

- **Unidade de previsão:** um registro de processo.
- **Alvo:** `Machine failure` (0 ou 1).
- **Uso do resultado:** sinal de apoio à priorização, não desligamento automático.
- **Erro perigoso:** falso negativo pode deixar passar um evento; muitos falsos positivos sobrecarregam a manutenção.
- **Métrica de acompanhamento:** recall, precisão, F1/PR-AUC e quantidade de alertas.

É um exercício educacional com dados sintéticos. Não é um modelo pronto para operação industrial.

## Fase 2 do CRISP-DM — Buscar e inspecionar o AI4I

UCI identifica o dataset pelo número 601. Veja as dimensões, nomes de colunas, primeiras linhas e alvos antes de escolher variáveis. A página oficial descreve `Machine failure` e também os modos `TWF`, `HDF`, `PWF`, `OSF` e `RNF`.

In [ ]:
ai4i = fetch_ucirepo(id=601)
X_bruto = ai4i.data.features.copy()
y_bruto = ai4i.data.targets.copy()

print("features:", X_bruto.shape)
print("targets:", y_bruto.shape)
print("colunas de entrada:", list(X_bruto.columns))
print("colunas de alvo:", list(y_bruto.columns))
display(X_bruto.head())
display(y_bruto.head())

### Como interpretar as colunas

`UID` e `Product ID` identificam registros/produtos; não são medições físicas para a previsão. Entradas candidatas incluem variante `Type`, temperatura do ar/processo, velocidade, torque e desgaste da ferramenta. `Machine failure` é o alvo. Os modos de falha são rótulos derivados do mesmo evento: não os inclua como entradas para prever `Machine failure`, pois isso revela a resposta.

## Fase 3 do CRISP-DM — Limpeza, alvo e prevenção de vazamento

As funções abaixo encontram nomes mesmo se a biblioteca devolver pequenas diferenças de formatação. Vamos selecionar somente entradas documentadas e excluir todos os alvos e IDs. Confira a lista impressa antes de prosseguir.

In [ ]:
def normalizar_nome(nome):
    return re.sub(r"[^a-z0-9]", "", str(nome).lower())

def localizar(colunas, prefixo_normalizado):
    encontrados = [c for c in colunas if normalizar_nome(c).startswith(prefixo_normalizado)]
    if not encontrados:
        raise KeyError(f"Não encontrei coluna iniciada por {prefixo_normalizado!r}: {list(colunas)}")
    return encontrados[0]

col_alvo = localizar(y_bruto.columns, "machinefailure")
col_type = localizar(X_bruto.columns, "type")
col_air = localizar(X_bruto.columns, "airtemperature")
col_process = localizar(X_bruto.columns, "processtemperature")
col_rpm = localizar(X_bruto.columns, "rotationalspeed")
col_torque = localizar(X_bruto.columns, "torque")
col_wear = localizar(X_bruto.columns, "toolwear")

colunas_modelo = [col_type, col_air, col_process, col_rpm, col_torque, col_wear]
X = X_bruto[colunas_modelo].copy()
y = y_bruto[col_alvo].astype(int).copy()
print("Entradas escolhidas:", colunas_modelo)
print("Alvo:", col_alvo)
print("IDs e modos de falha ficaram fora de X.")
display(X.head())

## Passo 4 — Exploração inicial

Verifique tipo, ausência, distribuição do alvo e proporção de falhas. Se a falha for rara, um modelo que sempre responde “sem falha” pode ter acurácia aparentemente alta e utilidade nula.

In [ ]:
display(X.dtypes.to_frame("tipo"))
display(X.isna().sum().to_frame("ausentes"))
contagem = y.value_counts().sort_index().rename(index={0:"sem falha", 1:"falha"})
display(contagem.to_frame("registros"))
display((y.value_counts(normalize=True).sort_index()*100).rename("percentual").to_frame())

ax = contagem.plot(kind="bar", title="AI4I: distribuição do alvo")
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade")
plt.show()

**Interpretação:** desbalanceamento muda como ler as métricas. Antes de escolher arquitetura, pergunte quantos alertas a equipe conseguiria verificar e qual tipo de erro tem maior custo.

## Passo 5 — Engenharia de atributos com justificativa

Podemos testar diferença entre temperatura de processo e ambiente e potência mecânica aproximada (`torque × velocidade angular`). São transformações calculáveis no instante da previsão. Não são leis completas da máquina; valide unidade, sensores e disponibilidade no cenário real.

In [ ]:
# Nomes localizados na etapa anterior; a velocidade está em rpm.
delta_temp = X[col_process] - X[col_air]
potencia_aprox_w = X[col_torque] * (X[col_rpm] * 2 * np.pi / 60)
X["delta_temp_K"] = delta_temp
X["potencia_aprox_W"] = potencia_aprox_w
display(X[["delta_temp_K", "potencia_aprox_W"]].describe())

## Passo 6 — Divisão treino, validação e teste

O AI4I é sintético e não fornece uma sequência temporal real por máquina. Para este exemplo tabular, faremos divisões estratificadas aleatórias e fixaremos a semente. Em telemetria temporal, prefira corte cronológico ou por equipamento; nunca misture janelas vizinhas aleatoriamente entre treino e teste.

In [ ]:
from sklearn.model_selection import train_test_split

X_treino_val, X_teste, y_treino_val, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    X_treino_val, y_treino_val, test_size=0.25, random_state=SEED, stratify=y_treino_val
)
print("treino:", X_treino.shape, "validação:", X_validacao.shape, "teste:", X_teste.shape)
print("taxa de falha:", round(y_treino.mean(), 4), round(y_validacao.mean(), 4), round(y_teste.mean(), 4))

## Fase 4 do CRISP-DM — Pipeline que evita vazamento de transformação

`OneHotEncoder` transforma `Type`; `StandardScaler` ajusta escala numérica. Ambos ficam dentro do `Pipeline`, portanto aprendem parâmetros apenas no treino durante `fit`. A MLP usa camadas ocultas `(32, 16)` como ponto de partida, não como arquitetura universal.

**Localiza??o da rede neural:** a MLP ? o modelo escolhido na Fase 4 (Modelagem). O pipeline prepara os dados e `fit` estima os pesos usando somente o treino. Ajustar hiperpar?metros e comparar modelos ? trabalho de modelagem; decidir se o resultado atende ao objetivo pertence ? Fase 5 (Avalia??o).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier

col_categorica = [col_type]
col_numericas = [c for c in X.columns if c not in col_categorica]
preparo = ColumnTransformer([
    ("categorica", OneHotEncoder(handle_unknown="ignore"), col_categorica),
    ("numerica", StandardScaler(), col_numericas),
])

mlp = Pipeline([
    ("preparo", preparo),
    ("modelo", MLPClassifier(
        hidden_layer_sizes=(32, 16), activation="relu", solver="adam",
        alpha=0.001, max_iter=400, early_stopping=True, random_state=SEED
    )),
])
mlp.fit(X_treino, y_treino)

Se o treinamento exibir aviso de convergência, não aumente camadas às cegas: confira escala, separação, taxa de aprendizado, número de iterações e baseline. `early_stopping` usa parte do treino para monitorar ajuste; o conjunto de teste continua intocado.

## Passo 8 — Baseline e métricas

Compare com Random Forest usando o mesmo preparo e partições. Reporte recall, precisão, F1, average precision (aproximação da área sob a curva precisão-recall) e matriz de confusão. Acurácia isolada não responde se a máquina com falha foi detectada.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             average_precision_score, balanced_accuracy_score)

baseline = Pipeline([
    ("preparo", preparo),
    ("modelo", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                      random_state=SEED, n_jobs=-1)),
])
baseline.fit(X_treino, y_treino)

for nome, modelo in [("MLP", mlp), ("Random Forest", baseline)]:
    pred = modelo.predict(X_validacao)
    prob = modelo.predict_proba(X_validacao)[:, 1]
    print("\n", nome)
    print("balanced accuracy:", round(balanced_accuracy_score(y_validacao, pred), 3))
    print("average precision:", round(average_precision_score(y_validacao, prob), 3))
    print(classification_report(y_validacao, pred, zero_division=0))
    print("matriz de confusão [linhas=real, colunas=previsto]:\n", confusion_matrix(y_validacao, pred))

### Como ler os resultados

- **Recall da falha:** fração das falhas detectadas; baixo recall significa eventos perdidos.
- **Precisão da falha:** fração dos alertas que eram falhas; baixa precisão gera inspeções desnecessárias.
- **Average precision:** resume a curva precisão-recall, útil quando a classe positiva é rara; compare sempre na mesma partição.
- **Balanced accuracy:** média do recall por classe, para não premiar apenas a classe majoritária.

O melhor modelo depende da capacidade de manutenção e do custo de cada erro.

## Fase 5 do CRISP-DM — Limiar de decisão escolhido na validação

`predict()` costuma usar limiar padrão 0,5. Em manutenção, o ponto adequado pode ser outro. Vamos escolher um limiar que maximize F2 na validação (dando mais peso ao recall) e só depois avaliar uma única vez no teste. Essa escolha é didática; um projeto real deveria incluir custo/capacidade e validação temporal.

In [ ]:
from sklearn.metrics import precision_recall_curve, fbeta_score

prob_validacao = mlp.predict_proba(X_validacao)[:, 1]
prec, rec, limiares = precision_recall_curve(y_validacao, prob_validacao)
f2 = (5 * prec[:-1] * rec[:-1]) / (4 * prec[:-1] + rec[:-1] + 1e-12)
limiar = limiares[int(np.argmax(f2))]
print("limiar escolhido na validação:", round(float(limiar), 3))
print("F2 na validação:", round(float(np.max(f2)), 3))

## Fase 1 do CRISP-DM0 — Avaliação final no teste

Não retreine nem ajuste o limiar após observar o resultado do teste nesta demonstração. Se o desempenho for fraco, volte ao ciclo de desenvolvimento e avalie em uma nova partição definida antes da comparação.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

prob_teste = mlp.predict_proba(X_teste)[:, 1]
pred_teste = (prob_teste >= limiar).astype(int)
print("limiar fixado:", round(float(limiar), 3))
print("precision:", round(precision_score(y_teste, pred_teste, zero_division=0), 3))
print("recall:", round(recall_score(y_teste, pred_teste, zero_division=0), 3))
print("F1:", round(f1_score(y_teste, pred_teste, zero_division=0), 3))
print("matriz de confusão:\n", confusion_matrix(y_teste, pred_teste))

## Fase 1 do CRISP-DM1 — Interpretar e comunicar, sem exagerar

Uma MLP não explica causalidade. Converta a saída em linguagem operacional com ressalvas: “o modelo atribuiu maior score a estes registros com base nos padrões do dataset sintético”. Não diga “a máquina vai falhar” nem use sozinho para decisão de segurança.

Documente dataset/versão, seleção de colunas, partição, limiar, métricas, limitações e responsável pelo acompanhamento.

## Segundo dataset público — MetroPT-3 para sequências

MetroPT-3 contém leituras multivariadas de sensores de compressor de uma unidade de produção de ar de trem. O UCI informa mais de 1,5 milhão de registros, 15 sinais, dados entre fevereiro e agosto de 2020 e registros de falhas reportadas; as linhas não têm um rótulo simples de falha por instante. O arquivo é grande, então a etapa é opcional e pode exigir memória e conexão.

### Passo 12 — Carregar e inspecionar MetroPT-3

Instale `ucimlrepo` se ainda não instalou. Não baixe novamente se a sessão já tem os dados. Confira nomes, tipos e intervalo temporal antes de modelar.

In [ ]:
# Opcional: pode baixar um arquivo de aproximadamente 208 MB.
metro = fetch_ucirepo(id=791)
metro_features = (metro.data.original.copy() if getattr(metro.data, "original", None) is not None else metro.data.features.copy())
print(metro_features.shape)
print(list(metro_features.columns))
display(metro_features.head())

O dataset é temporal e operacional, mas não vem com rótulo binário por linha. UCI lista relatórios de eventos de falha e recomenda usar o primeiro mês para treino e o restante para teste. Logo, podemos preparar previsão de sinal/anomalia; para classificação supervisionada é necessário construir rótulos temporais a partir dos relatórios, com regra explícita e revisão de domínio.

### Passo 13 — Preparar uma série ordenada

O bloco abaixo localiza timestamp e sinais de exemplo. Use apenas colunas que existirem no arquivo. Para manter o exercício manejável, selecione um período contínuo, preserve a ordem e verifique o intervalo de amostragem observado em vez de assumir frequência.

In [ ]:
def buscar_coluna(df, nome_normalizado):
    mapa = {normalizar_nome(c): c for c in df.columns}
    if nome_normalizado not in mapa:
        raise KeyError(f"Coluna {nome_normalizado!r} ausente. Colunas: {list(df.columns)}")
    return mapa[nome_normalizado]

col_ts = buscar_coluna(metro_features, "timestamp")
metro_df = metro_features.copy()
metro_df[col_ts] = pd.to_datetime(metro_df[col_ts], errors="coerce")
metro_df = metro_df.dropna(subset=[col_ts]).sort_values(col_ts)
print("período:", metro_df[col_ts].min(), "→", metro_df[col_ts].max())
print("diferenças entre leituras:")
display(metro_df[col_ts].diff().dropna().describe())

nomes_sensores = ["tp2", "tp3", "h1", "motorcurrent", "oiltemperature"]
col_sensores = [buscar_coluna(metro_df, n) for n in nomes_sensores]
metro_df[col_sensores] = metro_df[col_sensores].apply(pd.to_numeric, errors="coerce")
metro_df = metro_df.dropna(subset=col_sensores)
display(metro_df[[col_ts] + col_sensores].head())

### Passo 14 — Separação cronológica antes das janelas

Vamos demonstrar previsão do próximo valor de TP2 a partir de uma janela multissensor. O corte ocorre por data, nunca por embaralhamento. A escala é ajustada somente no trecho de treino. Para uma análise fiel ao protocolo do UCI, use um mês completo para treino e datas posteriores para teste; este código usa 80% cronológicos apenas como demonstração genérica.

In [ ]:
# Corte cronológico para demonstração. Para relatório formal, use a recomendação mensal do UCI.
corte_tempo = metro_df[col_ts].iloc[int(len(metro_df) * 0.8)]
treino_temporal = metro_df[metro_df[col_ts] < corte_tempo].copy()
teste_temporal = metro_df[metro_df[col_ts] >= corte_tempo].copy()
print("corte:", corte_tempo)
print("linhas treino/teste:", len(treino_temporal), len(teste_temporal))

from sklearn.preprocessing import StandardScaler
escala = StandardScaler()
treino_scaled = escala.fit_transform(treino_temporal[col_sensores])
teste_scaled = escala.transform(teste_temporal[col_sensores])

### Passo 15 — Criar janelas com formato para CNN/LSTM

Cada amostra usa 60 observações passadas de 5 sensores para prever TP2 no passo seguinte. Criamos janelas separadamente dentro de treino e teste para nenhuma janela atravessar o corte.

In [ ]:
def criar_janelas(matriz, indice_alvo, passos=60):
    X_seq, y_seq = [], []
    for fim in range(passos, len(matriz)):
        X_seq.append(matriz[fim-passos:fim, :])
        y_seq.append(matriz[fim, indice_alvo])
    return np.asarray(X_seq, dtype=np.float32), np.asarray(y_seq, dtype=np.float32)

idx_tp2 = col_sensores.index(buscar_coluna(metro_df, "tp2"))
X_seq_treino, y_seq_treino = criar_janelas(treino_scaled, idx_tp2, passos=60)
X_seq_teste, y_seq_teste = criar_janelas(teste_scaled, idx_tp2, passos=60)
print("forma treino:", X_seq_treino.shape, y_seq_treino.shape)
print("forma teste:", X_seq_teste.shape, y_seq_teste.shape)
# Esperado: (amostras, 60 passos, 5 canais)

### Passo 16 — Exemplo de CNN 1D (opcional)

Este exemplo requer TensorFlow. É um modelo pequeno de regressão para demonstrar a forma da entrada, não uma solução de manutenção pronta. Compare contra uma baseline que prevê o último valor de TP2. Para classificação/anomalia, o alvo e a avaliação precisam ser redefinidos.

In [ ]:
# Opcional: descomente após instalar TensorFlow em ambiente compatível.
# import tensorflow as tf
# modelo_cnn = tf.keras.Sequential([
#     tf.keras.layers.Input(shape=(X_seq_treino.shape[1], X_seq_treino.shape[2])),
#     tf.keras.layers.Conv1D(32, kernel_size=5, activation="relu"),
#     tf.keras.layers.MaxPooling1D(pool_size=2),
#     tf.keras.layers.Conv1D(16, kernel_size=3, activation="relu"),
#     tf.keras.layers.GlobalAveragePooling1D(),
#     tf.keras.layers.Dense(1),
# ])
# modelo_cnn.compile(optimizer="adam", loss="mae")
# historico = modelo_cnn.fit(X_seq_treino, y_seq_treino,
#     validation_split=0.2, epochs=10, batch_size=256, shuffle=False)
# pred_cnn = modelo_cnn.predict(X_seq_teste, verbose=0).ravel()
# baseline_ultimo_valor = X_seq_teste[:, -1, idx_tp2]
# print("MAE CNN:", np.mean(np.abs(y_seq_teste - pred_cnn)))
# print("MAE último valor:", np.mean(np.abs(y_seq_teste - baseline_ultimo_valor)))

## MLP, CNN 1D, LSTM ou autoencoder: decisão fundamentada

| Forma da entrada/pergunta | Candidato inicial | Comparação obrigatória |
|---|---|---|
| atributos agregados por registro/ciclo (AI4I) | MLP | baseline e Random Forest |
| sequências multissensor com padrões locais (MetroPT-3) | CNN 1D | último valor / estatísticas + modelo tabular |
| sequência longa com dependência temporal justificada | LSTM | CNN 1D e baseline temporal |
| dados normais abundantes, poucos rótulos | autoencoder exploratório | limites estatísticos e validação de eventos |

Autoencoder produz score de reconstrução, não causa. Em qualquer arquitetura, monitore deriva de sensores e versões do equipamento.

## Exercício final — miniestudo FactoryHub

Escolha uma das duas bases e escreva um plano reproduzível:

1. contexto e decisão a apoiar;
2. unidade de análise e alvo/score;
3. conjunto de entradas disponíveis no instante da decisão;
4. divisão de dados e justificativa;
5. baseline e arquitetura candidata;
6. métrica principal e custo de erro;
7. resultado esperado no dashboard/API e ressalvas ao operador.

Para AI4I, implemente ao menos a MLP e compare com Random Forest. Para MetroPT-3, construa as janelas e compare a CNN (se disponível) à baseline temporal. Não compare números de tarefas diferentes como se fossem a mesma competição.

## Checklist de qualidade e entrega

- [ ] colunas de ID e alvos derivados excluídas das entradas;
- [ ] treino/validação/teste adequados ao tipo de dados;
- [ ] preparo ajustado somente no treino;
- [ ] baseline, métrica e limiar documentados;
- [ ] dataset, licença/citação, limitações e versão registrados;
- [ ] score explicado como apoio à decisão, não diagnóstico automático.

Entregue o notebook executado ou relatório com células/saídas relevantes, comparação de modelo e recomendação de uso no FactoryHub.

## Fontes dos datasets e referências

- UCI. **AI4I 2020 Predictive Maintenance Dataset** (sintético; licença CC BY 4.0): <https://archive.ics.uci.edu/dataset/601/ai4i>. DOI: `10.24432/C5HS5C`.
- UCI. **MetroPT-3 Dataset** (série temporal de compressor; licença CC BY 4.0): <https://archive.ics.uci.edu/dataset/791/metropt+3+>. DOI: `10.24432/C5VW3R`.
- Scikit-learn. **MLP para classificação supervisionada**: <https://scikit-learn.org/stable/modules/neural_networks_supervised.html>.
- TensorFlow. **Previsão de séries temporais**: <https://www.tensorflow.org/tutorials/structured_data/time_series>.
- Matzka, S. **Explainable Artificial Intelligence for Predictive Maintenance Applications** (AI4I, 2020).